# Fireworks LLM Evaluation
This notebook loads `.env`, reads tasks/truth/results, and uses a Fireworks model as an LLM judge.

In [1]:
%pip install -q openai python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key=os.getenv("FIREWORKS_API_KEY")
base_url="https://api.fireworks.ai/inference/v1"
model=os.getenv("EVAL_MODEL") or os.getenv("MODEL") or os.getenv("ALLOWED_MODELS","accounts/fireworks/models/llama-v3p1-8b-instruct")

if "," in model:
    model=model.split(",")[0].strip()

client=OpenAI(api_key=api_key,base_url=base_url)

ROOT=Path("..")
OUT=ROOT/"output"
IN=ROOT/"input"
with open(IN/"tasks.json","r",encoding="utf-8") as f:
    tasks={x["task_id"]:x["prompt"] for x in json.load(f)}
with open(OUT/"truth.json","r",encoding="utf-8") as f:
    truth={x["task_id"]:x["answer"] for x in json.load(f)}
with open(OUT/"results.json","r",encoding="utf-8") as f:
    results={x["task_id"]:x["answer"] for x in json.load(f)}

print("Using model:",model)
print("Loaded",len(tasks),"tasks")


Using model: accounts/fireworks/models/minimax-m3
Loaded 8 tasks


In [ ]:
import json
import re

def judge(prompt, gt, pred):
    eval_prompt = f"""
You are an impartial evaluator.

Task:
{prompt}

Reference Answer:
{gt}

Student Answer:
{pred}

Score the student's answer from 0 to 100.

Return ONLY valid JSON.

{{
    "score": 95,
    "reason": "Brief explanation."
}}
"""

    r = client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": eval_prompt
            }
        ]
    )

    # Print the raw response for debugging
    print("=" * 80)
    print(r.model_dump_json(indent=2))
    print("=" * 80)

    msg = r.choices[0].message

    # Use content first, fall back to reasoning_content
    content = msg.content

    if content is None:
        content = getattr(msg, "reasoning_content", None)

    if content is None:
        raise RuntimeError(
            "The model returned neither content nor reasoning_content."
        )

    # Extract the first JSON object from the response
    match = re.search(r"\{.*\}", content, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON found in:\n{content}")

    return json.loads(match.group(0))

In [ ]:
print("Base URL:", base_url)
print("Model:", model)

rows = []

for tid, prompt in tasks.items():
    print(f"Evaluating {tid}...")

    ev = judge(prompt, truth[tid], results.get(tid, ""))

    rows.append({
        "task_id": tid,
        "score": ev["score"],
        "reason": ev["reason"]
    })

df = pd.DataFrame(rows)
display(df)

print("Average Score:", df["score"].mean())

df.to_csv("evaluation.csv", index=False)
print("Saved evaluation.csv")


Base URL: https://api.fireworks.ai/inference/v1
Model: accounts/fireworks/models/minimax-m3
Evaluating t1...
{
  "id": "chatcmpl-c3182efb422542b3830eab59b0421b5c",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\n    \"score\": 98,\n    \"reason\": \"The student's answer accurately and clearly covers all key points from the reference: definition of overfitting (learning training data too well including noise), the consequence (good on training, poor on new data), and why it matters (goal is generalization, not memorization). It also adds a useful extra point about unreliability in practice. The explanation is in simple terms as requested.\"\n}",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning_content": "The student's answer covers all the key points from the referenc

,task_id,score,reason
0,t1,98,The student's answer accurately and clearly co...
1,t2,100,The student's answer is correct. They properly...
2,t3,35,The review contains mixed sentiment (positive ...
3,t4,95,The student's answer accurately captures all k...
4,t5,100,The student's answer correctly identifies all ...
5,t6,95,The student correctly identified the typo bug ...
6,t7,100,The student's answer perfectly matches the ref...
7,t8,95,The student's solution is functionally correct...


Average Score: 89.75
Saved evaluation.csv
